In [1]:
import sys
sys.path.append('../src')

In [ ]:
from benchmark import SupportedBenchmarks, Benchmark
from endpoint_utils import Endpoint, SupportedEndpoints, fill_confusion_matrix
from experiment import start_experiment, Experiment
from nl_query import SupportedEmbedders
from llm_connector import SupportedModels, LLMApi
from prompting_utils import SupportedTemplates, fill_template
from example_selection import SupportedExampleSelections
from sparql_query import GeneratedSPARQLQuery
from prompting_utils import fill_template
from typing import List
from tqdm import tqdm
import datetime
import pickle
import os

In [3]:
endpoint = Endpoint(SupportedEndpoints.DBPEDIA_2016_04)

In [ ]:
def save_experiments(experiments: List[Experiment]):
    for experiment in experiments:
        with open(f'../data/persistence/LCQUAD/experiments/{experiment.id}.pkl', 'wb') as dump_file:
            pickle.dump(experiment, dump_file)

# Load Experiments

In [ ]:
experiments = []

for file in os.listdir('../data/persistence/LCQUAD/experiments/'):
    if file.endswith('.pkl'):
        file_path = os.path.join('../data/persistence/LCQUAD/experiments/', file)
        with open(file_path, 'rb') as file_pkl:
            try:
                exp_object = pickle.load(file_pkl)
                print('----')
                print(f'Description: {exp_object.description}')
                print(f'File: {file_pkl}')
                print('----')
                experiments.append(exp_object)
            except Exception as e:
                print(f"Error loading experiments: {e}")
                
print(f"Total of loaded objects: {len(experiments)}")

# Generating queries using LLMApi

In [ ]:
count_exp = 0
count_round = 0
for experiment in experiments:
    try:
        for round in experiment.rounds:
            for execution in tqdm(round.executions, desc=f"Experiment {count_exp}, Round {count_round}"):
                if execution.generated_query == None:
                    timestamp_beginning = datetime.datetime.now()
                    llm = LLMApi(model = experiment.model)
                    llm_output = llm.get_model_response(input_text = execution.prompt)
                    execution.generated_query = GeneratedSPARQLQuery(sparql_query=llm_output, gold_query=execution.gold_query)
                    timestamp_end = datetime.datetime.now()
                    execution.sec_api_latency = (timestamp_end-timestamp_beginning).total_seconds
                    execution.exp_ts = datetime.datetime.now()
            count_round += 1
    except Exception as e:
        save_experiments(experiments)
        print(f'error: {e}')
        count_round = 0
        continue
    finally:
        count_round = 0
        save_experiments(experiments)
    count_exp +=1


### Getting the endpoint output

In [ ]:
import time

count_exp = 0
count_round = 0

# Tempo máximo em segundos (5 minutos)
max_time = 5 * 60

for experiment in experiments:
    for round in experiment.rounds:
        #start_time = time.time()  # Inicia o temporizador
        for execution in tqdm(round.executions, desc=f"Experiment {count_exp}, Round {count_round}"):
            #if time.time() - start_time > max_time:
                #print(f"Tempo máximo de execução de 5 minutos excedido para Experimento {count_exp}, Rodada {count_round}")
                #break
            if execution.generated_query.endpoint_output == None and not execution.generated_query.has_query_parsing_error:
                execution.generated_query.endpoint_output = endpoint.get_endpoint_output(execution.generated_query)
        count_round += 1
    count_exp += 1
    count_round = 0

### Getting Hallucinated URIs

In [ ]:
count_exp = 0
count_round = 0

for experiment in experiments:
    for round in experiment.rounds:
        for execution in tqdm(round.executions, desc=f"Experiment {count_exp}, Round {count_round}"):
            execution.generated_query.hallucinated_uris = endpoint.get_hallucinated_uris(execution.generated_query)
        count_round += 1
    count_exp += 1
    count_round = 0

### Filling Confusion Matrixes and updating experiments data

In [ ]:
count_exp = 0
count_round = 0
for experiment in experiments:
    for round in experiment.rounds:
        for execution in tqdm(round.executions, desc=f"Experiment {count_exp}, Round {count_round}"):
            if execution.generated_query.confusion_matrix == None:
                execution.generated_query.confusion_matrix = fill_confusion_matrix(execution.generated_query)
        count_round += 1
    count_exp += 1
    count_round = 0

In [7]:
save_experiments(experiments)